# 05 — Final Load Prep for Tableau

**Objective:** Prepare aggregated summary tables and KPI datasets for Tableau dashboard creation.

**Outputs:** Multiple CSV files in `data/processed/` ready for Tableau import.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/cleaned_data.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'Loaded {len(df):,} rows')

## 5.1 KPI Summary Table

In [ ]:
total_rev = df['Revenue'].sum()
order_vals = df.groupby('InvoiceNo')['Revenue'].sum()
avg_order = order_vals.mean()
cust_orders = df.groupby('CustomerID')['InvoiceNo'].nunique()
repeat_rate = (cust_orders > 1).sum() / len(cust_orders) * 100
total_customers = df['CustomerID'].nunique()
total_orders = df['InvoiceNo'].nunique()
total_products = df['StockCode'].nunique()

kpis = pd.DataFrame({
    'KPI': ['Total Revenue', 'Average Order Value', 'Total Customers',
            'Total Orders', 'Unique Products', 'Repeat Customer Rate',
            'Total Transactions'],
    'Value': [f'£{total_rev:,.2f}', f'£{avg_order:,.2f}', f'{total_customers:,}',
              f'{total_orders:,}', f'{total_products:,}', f'{repeat_rate:.1f}%',
              f'{len(df):,}']
})
print(kpis.to_string(index=False))
kpis.to_csv('../data/processed/kpi_summary.csv', index=False)
print('\nSaved kpi_summary.csv')

## 5.2 Monthly Revenue Summary

In [ ]:
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)
monthly = df.groupby('YearMonth').agg(
    TotalRevenue=('Revenue', 'sum'),
    TotalOrders=('InvoiceNo', 'nunique'),
    TotalCustomers=('CustomerID', 'nunique'),
    TotalQuantity=('Quantity', 'sum'),
    AvgOrderValue=('Revenue', lambda x: x.sum() / df.loc[x.index, 'InvoiceNo'].nunique())
).reset_index()
monthly = monthly.round(2)
print(monthly.to_string(index=False))
monthly.to_csv('../data/processed/monthly_revenue_summary.csv', index=False)
print('\nSaved monthly_revenue_summary.csv')

## 5.3 Product Summary

In [ ]:
product = df.groupby(['StockCode', 'Description']).agg(
    TotalRevenue=('Revenue', 'sum'),
    TotalQuantity=('Quantity', 'sum'),
    OrderCount=('InvoiceNo', 'nunique'),
    CustomerCount=('CustomerID', 'nunique')
).reset_index().sort_values('TotalRevenue', ascending=False).round(2)
print(f'Total products: {len(product):,}')
print(product.head(10).to_string(index=False))
product.to_csv('../data/processed/product_summary.csv', index=False)
print('\nSaved product_summary.csv')

## 5.4 Country Summary

In [ ]:
country = df.groupby('Country').agg(
    TotalRevenue=('Revenue', 'sum'),
    TotalCustomers=('CustomerID', 'nunique'),
    TotalOrders=('InvoiceNo', 'nunique'),
    TotalQuantity=('Quantity', 'sum')
).reset_index().sort_values('TotalRevenue', ascending=False).round(2)
print(country.to_string(index=False))
country.to_csv('../data/processed/country_summary.csv', index=False)
print('\nSaved country_summary.csv')

## 5.5 Customer Summary

In [ ]:
customer = df.groupby('CustomerID').agg(
    TotalRevenue=('Revenue', 'sum'),
    OrderCount=('InvoiceNo', 'nunique'),
    TotalQuantity=('Quantity', 'sum'),
    AvgOrderValue=('Revenue', 'mean'),
    FirstPurchase=('InvoiceDate', 'min'),
    LastPurchase=('InvoiceDate', 'max'),
    Country=('Country', 'first'),
    UniqueProducts=('StockCode', 'nunique')
).reset_index().round(2)
customer['IsRepeat'] = (customer['OrderCount'] > 1).astype(int)
median_rev = customer['TotalRevenue'].median()
customer['Segment'] = np.where(customer['TotalRevenue'] >= median_rev, 'High Value', 'Low Value')
print(f'Total customers: {len(customer):,}')
print(customer.head(10).to_string(index=False))
customer.to_csv('../data/processed/customer_summary.csv', index=False)
print('\nSaved customer_summary.csv')

## 5.6 Day/Hour Heatmap Data

In [ ]:
heatmap = df.groupby(['DayOfWeek', 'Hour'])['Revenue'].sum().reset_index()
heatmap.to_csv('../data/processed/day_hour_revenue.csv', index=False)
print('Saved day_hour_revenue.csv')

## 5.7 Verify All Outputs

In [ ]:
processed_dir = '../data/processed/'
print('Files in data/processed/:')
for f in sorted(os.listdir(processed_dir)):
    size = os.path.getsize(os.path.join(processed_dir, f))
    print(f'  {f:40s} {size/1024:8.1f} KB')

## Summary

All summary tables have been exported and are ready for Tableau:

| File | Description |
|------|-------------|
| `cleaned_data.csv` | Full cleaned transactional dataset |
| `kpi_summary.csv` | Key performance indicators |
| `monthly_revenue_summary.csv` | Revenue aggregated by month |
| `product_summary.csv` | Revenue/quantity by product |
| `country_summary.csv` | Revenue/customers by country |
| `customer_summary.csv` | Per-customer metrics + segmentation |
| `day_hour_revenue.csv` | Revenue by day of week and hour |